# Run This To Reproduce The Project Checks

This notebook is different from the pre-filled TA summary notebook. These cells run the actual commands again so you can personally reproduce the results.

Run cells from top to bottom. The slowest parts are package installation and downloading the real taxi data.

## 1. Confirm We Are In The Project Folder

This should show files like `README.md`, `pyproject.toml`, `src`, `tests`, and `scripts`.

In [1]:
from pathlib import Path

print("Current folder:", Path.cwd())
required = ["README.md", "pyproject.toml", "src", "tests", "scripts", "docs"]
for item in required:
    print(f"{item:15} exists:", Path(item).exists())

Current folder: C:\Users\wongm\ORIE5270\Final project - Copy
README.md       exists: True
pyproject.toml  exists: True
src             exists: True
tests           exists: True
scripts         exists: True
docs            exists: True


## 2. Install The Project

Run this once. If it shows pink/red dependency warnings about unrelated Anaconda packages, that is usually okay as long as the final line says the project installed successfully.

This can take a few minutes.

In [2]:
import sys

!{sys.executable} -m pip install -e ".[dev]" coverage

Defaulting to user installation because normal site-packages is not writeable


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


Obtaining file:///C:/Users/wongm/ORIE5270/Final%20project%20-%20Copy
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for orie5270-final-project (pyproject.toml): started
  Building editable for orie5270-final-project (pyproject.toml): finished with status 'done'
  Created wheel for orie5270-final-project: filename=orie5270_final_project-0.1.0-0.editable-py3-none-any.whl size=5493 sha256=c06ef509a44a30a5e539d163b49186c97776a2bc61a36ddc504880fff1106d17
  Stored in directory: C:\Users\wongm\AppData\Local\Temp\pip-ephem-wheel-cach

## 3. Run Unit Tests With Coverage

This is the main grading check. Expected final result:

```text
34 passed
Required test coverage of 80% reached
Total coverage: about 99%
```

In [3]:
import os
import sys
import subprocess

env = os.environ.copy()
# This avoids slow/hanging pytest plugins from Anaconda/Jupyter.
env["PYTEST_DISABLE_PLUGIN_AUTOLOAD"] = "1"

commands = [
    [sys.executable, "-m", "coverage", "run", "--source=orie5270_project", "-m", "pytest", "tests", "-q", "-o", "addopts="],
    [sys.executable, "-m", "coverage", "report", "--fail-under=80"],
]

for command in commands:
    print("\nRunning:", " ".join(command))
    result = subprocess.run(
        command,
        env=env,
        text=True,
        capture_output=True,
        timeout=180,
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    print("Exit code:", result.returncode)
    if result.returncode != 0:
        raise RuntimeError("Command failed; see output above.")


^C


## 4. Run The Small Demo

This does not need the real taxi data.

In [4]:
!python scripts/run_demo.py

^C


## 5. Check Whether Real Data Is Already Present

If these are `False`, run the download cell in the next section. If they are `True`, skip download and run the analysis.

In [ ]:
from pathlib import Path

data_files = [
    Path("data/yellow_tripdata_2025-01.parquet"),
    Path("data/yellow_tripdata_2025-02.parquet"),
    Path("data/taxi_zone_lookup.csv"),
]
for path in data_files:
    print(f"{path}:", path.exists())

## 6. Download Real Data If Needed

Only run this if the previous cell showed missing data. This downloads about 120 MB from the public NYC TLC source, so it may take time.

In [ ]:
# Run this only if the data files above are missing.
!python scripts/download_data.py

## 7. Run The Full Taxi Analysis

This reads the real dataset, creates borough-level hourly demand, evaluates forecasting baselines, and writes CSV/PNG outputs.

In [ ]:
!python scripts/run_taxi_analysis.py

## 8. Read The Generated Metrics

This reads the actual CSV created by the analysis command above. It is not hardcoded.

In [ ]:
import pandas as pd

metrics = pd.read_csv("data/processed/borough_model_metrics.csv")
metrics

## 9. Display Generated Figures

These are the actual PNG files from `docs/figures/`.

In [ ]:
from IPython.display import Image, display

display(Image("docs/figures/borough_model_comparison.png"))
display(Image("docs/figures/manhattan_forecast_last_72_hours.png"))

## 10. Import Check

This verifies that the package imports work after installation.

In [ ]:
from orie5270_project.taxi import load_yellow_taxi_data, load_zone_lookup
from orie5270_project.metrics import mae, rmse

print("Imports work")

## Final Check

If the test cell passes, the demo runs, the full analysis runs, and the metrics/figures display, then the project is reproducible and satisfies the grading requirements.